# Fix BEN share class names missing the word "Shares" (bcgov/entity#33621)

Companies altered from LTD to BEN in COLIN were migrated to LEAR by an early notebook that dropped the
trailing `Shares` from every share class name (e.g. `Class A Common` instead of `Class A Common Shares`).

For each impacted business this notebook files a **staff Correction** against the LTD-to-BEN **Alteration**
filing through legal-api, exactly as staff would through the UI:

- every share class (and series) name gets ` Shares` appended;
- the ledger shows `Correction for Alteration filed on <date>` with the Filing Detail
  `Share class(es) corrected to reflect the word ‘shares’ at the end of name`;
- `correction.type = STAFF` so the ledger records it as a staff error;
- the fee is waived (`header.waiveFees = true`).

Going through legal-api (instead of raw SQL) keeps versioning, ledger, outputs and COLIN sync consistent.

**Safety:** `DRY_RUN=true` (default) only reports and prints payloads. Set `DRY_RUN=false` to submit.
Restrict the run to the business list from Ops with `IDENTIFIERS=BC1234567,BC7654321`.

**Prerequisites**
- read access to the LEAR database (used only to identify impacted businesses and verify afterwards);
- a staff JWT (`LEGAL_API_TOKEN`) whose user can file corrections and use staff payment (`waiveFees`);
- `ACCOUNT_ID` of the staff account the filing is submitted under.


In [ ]:
%pip install pandas
%pip install sqlalchemy>=2.0
%pip install python-dotenv
%pip install requests

## Configuration

In [ ]:
import json
import os
import time
from datetime import date

import pandas as pd
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

load_dotenv()

LEGAL_API_URL = os.getenv('LEGAL_API_URL', '').rstrip('/')
LEGAL_API_TOKEN = os.getenv('LEGAL_API_TOKEN')
LEGAL_API_KEY = os.getenv('LEGAL_API_KEY')  # optional, gateway x-apikey
ACCOUNT_ID = os.getenv('ACCOUNT_ID')
CERTIFIED_BY = os.getenv('CERTIFIED_BY')
DRY_RUN = os.getenv('DRY_RUN', 'true').strip().lower() != 'false'
IDENTIFIERS = [x.strip().upper() for x in os.getenv('IDENTIFIERS', '').split(',') if x.strip()]
OUTPUT_DIR = os.getenv('OUTPUT_DIR', '../generated')

CORRECTION_COMMENT = 'Share class(es) corrected to reflect the word ‘shares’ at the end of name'
SHARE_NAME_SUFFIX = ' Shares'
RESERVED_WORDS_CLASS = {'share', 'shares', 'value'}
RESERVED_WORDS_SERIES = {'share', 'shares'}

for name, value in {'LEGAL_API_URL': LEGAL_API_URL, 'LEGAL_API_TOKEN': LEGAL_API_TOKEN,
                    'ACCOUNT_ID': ACCOUNT_ID, 'CERTIFIED_BY': CERTIFIED_BY}.items():
    if not value:
        raise ValueError(f'{name} is not set')

HEADERS = {
    'Authorization': f'Bearer {LEGAL_API_TOKEN}',
    'Account-Id': ACCOUNT_ID,
    'Content-Type': 'application/json',
}
if LEGAL_API_KEY:
    HEADERS['x-apikey'] = LEGAL_API_KEY

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'DRY_RUN={DRY_RUN}  legal-api={LEGAL_API_URL}  identifiers filter={IDENTIFIERS or "none (all impacted)"}')

In [ ]:
lear_uri = (
    f"postgresql://{os.getenv('DATABASE_LEAR_USERNAME')}:{os.getenv('DATABASE_LEAR_PASSWORD')}"
    f"@{os.getenv('DATABASE_LEAR_HOST')}:{os.getenv('DATABASE_LEAR_PORT')}/{os.getenv('DATABASE_LEAR_NAME')}"
)
try:
    lear_engine = create_engine(lear_uri)
    with lear_engine.connect() as conn:
        conn.execute(text('SELECT 1'))
    print('LEAR database connection OK')
except SQLAlchemyError as e:
    print(f'LEAR database connection failed: {e}')
    raise

## Identify impacted businesses

BEN businesses with a completed Alteration whose share class (or series) name does not end in ` Shares`.
The Alteration to correct is the earliest completed one that set the legal type to BEN
(falls back to the earliest completed Alteration). Review the table before submitting anything.

In [ ]:
IMPACTED_QUERY = '''
WITH bad_names AS (
    SELECT b.id AS business_id, b.identifier, 'class' AS kind, sc.id AS record_id, sc.name
    FROM businesses b
    JOIN share_classes sc ON sc.business_id = b.id
    WHERE b.legal_type = 'BEN'
      AND sc.name NOT LIKE '% Shares'
    UNION ALL
    SELECT b.id, b.identifier, 'series', ss.id, ss.name
    FROM businesses b
    JOIN share_classes sc ON sc.business_id = b.id
    JOIN share_series ss ON ss.share_class_id = sc.id
    WHERE b.legal_type = 'BEN'
      AND ss.name NOT LIKE '% Shares'
),
alteration AS (
    SELECT DISTINCT ON (f.business_id)
           f.business_id, f.id AS alteration_filing_id, f.filing_date::date AS alteration_filing_date, f.source
    FROM filings f
    WHERE f.filing_type = 'alteration'
      AND f.status = 'COMPLETED'
      AND f.business_id IN (SELECT business_id FROM bad_names)
    ORDER BY f.business_id,
             (f.filing_json->'filing'->'alteration'->'business'->>'legalType' = 'BEN') DESC NULLS LAST,
             f.filing_date ASC
)
SELECT bn.identifier, bn.kind, bn.record_id, bn.name,
       a.alteration_filing_id, a.alteration_filing_date, a.source AS alteration_source
FROM bad_names bn
JOIN alteration a ON a.business_id = bn.business_id
ORDER BY bn.identifier, bn.kind, bn.record_id
'''

impacted_df = pd.read_sql(IMPACTED_QUERY, lear_engine)
if IDENTIFIERS:
    missing = sorted(set(IDENTIFIERS) - set(impacted_df['identifier']))
    if missing:
        print(f'WARNING: {len(missing)} identifiers from IDENTIFIERS are not impacted (or have no completed alteration): {missing}')
    impacted_df = impacted_df[impacted_df['identifier'].isin(IDENTIFIERS)]

business_ids = sorted(impacted_df['identifier'].unique())
print(f'{len(impacted_df)} share class/series names across {len(business_ids)} businesses')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(impacted_df)

## Build correction payloads

Share classes are fetched from legal-api so each class/series keeps its `id` and the filer updates the
existing rows in place instead of recreating them. `resolutionDates` is omitted so existing resolutions
are untouched.

In [ ]:
def fix_share_name(name: str) -> str:
    name = (name or '').strip()
    if name.endswith(SHARE_NAME_SUFFIX):
        return name
    if name.lower().endswith(' shares'):
        name = name[:-len(' shares')].rstrip()
    return f'{name}{SHARE_NAME_SUFFIX}'


def has_reserved_words(fixed_name: str, reserved: set) -> bool:
    words = fixed_name[:-len(SHARE_NAME_SUFFIX)].lower().split()
    return any(w in reserved for w in words)


def api_get(path: str) -> dict:
    resp = requests.get(f'{LEGAL_API_URL}{path}', headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return resp.json()


def build_share_classes(identifier: str) -> tuple[list, list, list]:
    """Return (corrected share classes, change log rows, problems)."""
    share_classes = api_get(f'/businesses/{identifier}/share-classes')['shareClasses']
    changes, problems = [], []
    for share_class in share_classes:
        old = share_class['name']
        new = fix_share_name(old)
        if has_reserved_words(new, RESERVED_WORDS_CLASS):
            problems.append(f'class {old!r} contains a reserved word')
        if old != new:
            changes.append({'identifier': identifier, 'kind': 'class', 'id': share_class['id'], 'old': old, 'new': new})
        share_class['name'] = new
        for series in share_class.get('series', []):
            old_s = series['name']
            new_s = fix_share_name(old_s)
            if has_reserved_words(new_s, RESERVED_WORDS_SERIES):
                problems.append(f'series {old_s!r} contains a reserved word')
            if old_s != new_s:
                changes.append({'identifier': identifier, 'kind': 'series', 'id': series['id'], 'old': old_s, 'new': new_s})
            series['name'] = new_s
    return share_classes, changes, problems


def build_correction_filing(identifier: str, business: dict, alteration_filing_id: int,
                            alteration_filing_date, share_classes: list) -> dict:
    return {
        'filing': {
            'header': {
                'name': 'correction',
                'date': date.today().isoformat(),
                'certifiedBy': CERTIFIED_BY,
                'waiveFees': True,
            },
            'business': {
                'identifier': identifier,
                'legalType': business['legalType'],
                'legalName': business['legalName'],
            },
            'correction': {
                'type': 'STAFF',
                'correctedFilingId': int(alteration_filing_id),
                'correctedFilingType': 'alteration',
                'correctedFilingDate': str(alteration_filing_date),
                'comment': CORRECTION_COMMENT,
                'shareStructure': {'shareClasses': share_classes},
            },
        }
    }


payloads, change_rows, skipped = {}, [], []
for identifier in business_ids:
    row = impacted_df[impacted_df['identifier'] == identifier].iloc[0]
    try:
        business = api_get(f'/businesses/{identifier}')['business']
        if business.get('state') != 'ACTIVE' or business.get('legalType') != 'BEN':
            skipped.append({'identifier': identifier, 'reason': f"state={business.get('state')} legalType={business.get('legalType')}"})
            continue
        share_classes, changes, problems = build_share_classes(identifier)
        if problems:
            skipped.append({'identifier': identifier, 'reason': '; '.join(problems)})
            continue
        if not changes:
            skipped.append({'identifier': identifier, 'reason': 'no share name changes needed'})
            continue
        payloads[identifier] = build_correction_filing(
            identifier, business, row['alteration_filing_id'], row['alteration_filing_date'], share_classes)
        change_rows.extend(changes)
    except requests.HTTPError as e:
        skipped.append({'identifier': identifier, 'reason': f'{e.response.status_code}: {e.response.text[:300]}'})

changes_df = pd.DataFrame(change_rows)
skipped_df = pd.DataFrame(skipped)
print(f'{len(payloads)} businesses ready, {len(changes_df)} name changes, {len(skipped_df)} skipped')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(changes_df)
    if not skipped_df.empty:
        print('SKIPPED:')
        display(skipped_df)

## Submit corrections

With `DRY_RUN=true` this only prints the payloads. Set `DRY_RUN=false` in `.env`, re-run the configuration
cell and this cell to file them. Results are written to `OUTPUT_DIR`.

In [ ]:
results = []
for identifier, payload in payloads.items():
    if DRY_RUN:
        print(f'--- DRY RUN {identifier} ---')
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        continue
    resp = requests.post(f'{LEGAL_API_URL}/businesses/{identifier}/filings',
                         headers=HEADERS, json=payload, timeout=120)
    body = resp.json() if resp.content else {}
    filing_id = body.get('filing', {}).get('header', {}).get('filingId')
    status = body.get('filing', {}).get('header', {}).get('status')
    results.append({'identifier': identifier, 'http_status': resp.status_code, 'filing_id': filing_id,
                    'status': status, 'error': None if resp.ok else resp.text[:500]})
    print(f'{identifier}: HTTP {resp.status_code} filing_id={filing_id} status={status}')

results_df = pd.DataFrame(results)
if not results_df.empty:
    out = os.path.join(OUTPUT_DIR, f'fix_share_class_names_{date.today().isoformat()}.csv')
    results_df.to_csv(out, index=False)
    print(f'results written to {out}')
    display(results_df)

## Verify

Waits for the filer to complete each correction, then re-runs the identification query for the
submitted businesses. The second table must be empty.

In [ ]:
if not DRY_RUN and not results_df.empty:
    pending = results_df[results_df['http_status'].between(200, 299)]['identifier'].tolist()
    statuses = {}
    deadline = time.time() + 15 * 60
    while pending and time.time() < deadline:
        for identifier in list(pending):
            filing_id = int(results_df.loc[results_df['identifier'] == identifier, 'filing_id'].iloc[0])
            status = api_get(f'/businesses/{identifier}/filings/{filing_id}')['filing']['header']['status']
            statuses[identifier] = status
            if status in ('COMPLETED', 'ERROR', 'PENDING_CORRECTION'):
                pending.remove(identifier)
        if pending:
            time.sleep(15)
    print(pd.Series(statuses, name='status').value_counts())
    if pending:
        print(f'still not completed after 15 min: {pending}')

    submitted = [f"'{i}'" for i in results_df['identifier']]
    remaining_df = pd.read_sql(
        IMPACTED_QUERY.replace('ORDER BY bn.identifier', f'AND bn.identifier IN ({",".join(submitted)}) ORDER BY bn.identifier'),
        lear_engine)
    print(f'{len(remaining_df)} share names still missing the suffix for submitted businesses (expected 0)')
    display(remaining_df)